In [2]:
# Transition to a structured workflow when your task requires deterministic control, conditional branching, 
# or handling complex multi-step processes with cycles.

# DiGraphBuilder is a fluent utility that lets you easily construct execution graphs for workflows. It supports building:

# 1. Sequential chains

# 2. Parallel fan-outs

# 3. Conditional branching

# 4. Loops with safe exit conditions

# Each node in the graph represents an agent, and edges define the allowed execution paths. 
# Edges can optionally have conditions based on agent messages.

In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent,UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="gemini-2.5-flash",
    api_key=api_key,
)

Sequential Flow

In [2]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

writer = AssistantAgent(
    name="Writer",
    description="A writer agent that generates text based on user input.",
    model_client=model_client,
    system_message="You are a creative writer. Please write a story based on the user's input.",
)

reviewer = AssistantAgent(
    name="Reviewer",
    description="A reviewer agent that provides feedback on the text generated by the writer.",
    model_client=model_client,
    system_message="You are a reviewer. Please provide feedback on the text generated by the writer.",
)

In [7]:
builder = DiGraphBuilder()
builder.add_node(writer).add_node(reviewer)
builder.add_edge(writer, reviewer)

graph = builder.build()

In [8]:
graph

DiGraph(nodes={'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)

In [9]:
team = GraphFlow([writer,reviewer], graph)

In [10]:
stream = team.run_stream(task ="Write a good poem about India in less than 30 words.")

async for event in stream:
    print(event)

id='d5c2a2e9-4958-47d6-b9b7-64763b2570ce' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 12, 8, 16, 42, 5, 669785, tzinfo=datetime.timezone.utc) content='Write a good poem about India in less than 30 words.' type='TextMessage'
id='dbf6d179-a7aa-4b36-be46-49eccb88d617' source='Writer' models_usage=RequestUsage(prompt_tokens=34, completion_tokens=32) metadata={} created_at=datetime.datetime(2025, 12, 8, 16, 42, 12, 36409, tzinfo=datetime.timezone.utc) content="Spices scent the ancient air,\nGanges whispers, mountains soar.\nA vibrant tapestry, beyond compare,\nIndia's spirit, we adore." type='TextMessage'
id='29fbeacd-ef3c-45d9-bd28-6a3c128cc316' source='Reviewer' models_usage=RequestUsage(prompt_tokens=65, completion_tokens=206) metadata={} created_at=datetime.datetime(2025, 12, 8, 16, 42, 16, 271665, tzinfo=datetime.timezone.utc) content='This is an excellent poem, especially considering the strict word limit!\n\nHere\'s why it\'s good:\n\n*   **Concise 

Parallel Flow

In [3]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient

agent_a = AssistantAgent("A", model_client=model_client, system_message="You are a helpful assistant.")
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to Chinese.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Japanese.")
# Create a directed graph with fan-out flow A -> (B, C).
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
builder.add_edge(agent_a, agent_b).add_edge(agent_a, agent_c)
graph = builder.build()
# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)
# Run the team and print the events.
async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)




id='24f8250c-c21e-4b5c-a3c6-02ba0703a25a' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 12, 8, 16, 56, 37, 95218, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='c9d18c3e-8b94-46d5-9591-2f668f9375f4' source='A' models_usage=RequestUsage(prompt_tokens=16, completion_tokens=610) metadata={} created_at=datetime.datetime(2025, 12, 8, 16, 56, 48, 21397, tzinfo=datetime.timezone.utc) content='The first hint of dawn wasn\'t a sound or a sight for Jasper, but a *feeling*. A subtle shift in the air, a cooling of the floor, a premonition that the golden light was coming. He uncurled himself from a tight ball on the worn armchair, stretching with a slow, luxurious languor that began with his front paws splayed, arching his back, and ended with a shimmy of his tail.\n\nHis human, Eleanor, was still a lump under the duvet. Jasper acknowledged her presence with a soft, questioning meow that was more a rumble in his ches

Message Filtering

In GraphFlow, the execution graph is defined using DiGraph, which controls the order in which agents execute. 
However, the execution graph does not control what messages an agent receives from other agents. 
By default, all messages are sent to all agents in the graph.

Message filtering is a separate feature that allows you to filter the messages received by each agent and limiting their 
model context to only the relevant information. The set of message filters defines the message graph in the flow.

Specifying the message graph can help with:

1. Reduce hallucinations

2. Control memory load

3. Focus agents only on relevant information

You can use MessageFilterAgent together with MessageFilterConfig and PerSourceFilter to define these rules.

In [4]:
from autogen_agentchat.agents import AssistantAgent, MessageFilterAgent, MessageFilterConfig, PerSourceFilter
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Model client
client = model_client

# Create agents
researcher = AssistantAgent(
    "researcher", model_client=client, system_message="Summarize key facts about climate change."
)
analyst = AssistantAgent("analyst", model_client=client, system_message="Review the summary and suggest improvements.")
presenter = AssistantAgent(
    "presenter", model_client=client, system_message="Prepare a presentation slide based on the final summary."
)

# Apply message filtering
filtered_analyst = MessageFilterAgent(
    name="analyst",
    wrapped_agent=analyst,
    filter=MessageFilterConfig(per_source=[PerSourceFilter(source="researcher", position="last", count=1)]),
)

filtered_presenter = MessageFilterAgent(
    name="presenter",
    wrapped_agent=presenter,
    filter=MessageFilterConfig(per_source=[PerSourceFilter(source="analyst", position="last", count=1)]),
)

# Build the flow
builder = DiGraphBuilder()
builder.add_node(researcher).add_node(filtered_analyst).add_node(filtered_presenter)
builder.add_edge(researcher, filtered_analyst).add_edge(filtered_analyst, filtered_presenter)

# Create the flow
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=builder.build(),
)

# Run the flow
await Console(flow.run_stream(task="Summarize key facts about climate change."))

---------- TextMessage (user) ----------
Summarize key facts about climate change.
---------- TextMessage (researcher) ----------
Here are the key facts about climate change:

1.  **Definition:** Climate change refers to long-term shifts in temperatures and weather patterns globally. While some shifts are natural, the current rapid change is predominantly driven by human activities.

2.  **Primary Cause: Human Activity:** The burning of fossil fuels (coal, oil, and natural gas) for energy, transportation, and industry is the main driver. Other significant human contributions include deforestation, agriculture, and industrial processes.

3.  **Greenhouse Gas Effect:** These human activities release large amounts of greenhouse gases (GHGs) – primarily carbon dioxide (CO2), but also methane (CH4) and nitrous oxide (N2O) – into the atmosphere. These gases trap heat, leading to a warming effect (the enhanced greenhouse effect).

4.  **Observed Warming:** The Earth's average global temperatu

TaskResult(messages=[TextMessage(id='4e14dae0-6f43-4178-bbe6-08d686c50a18', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 17, 57, 686822, tzinfo=datetime.timezone.utc), content='Summarize key facts about climate change.', type='TextMessage'), TextMessage(id='e309cb77-cffe-42f7-95f0-a5b644880f58', source='researcher', models_usage=RequestUsage(prompt_tokens=18, completion_tokens=678), metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 18, 7, 151640, tzinfo=datetime.timezone.utc), content="Here are the key facts about climate change:\n\n1.  **Definition:** Climate change refers to long-term shifts in temperatures and weather patterns globally. While some shifts are natural, the current rapid change is predominantly driven by human activities.\n\n2.  **Primary Cause: Human Activity:** The burning of fossil fuels (coal, oil, and natural gas) for energy, transportation, and industry is the main driver. Other significant human contributi

Advanced Example: Conditional Loop + Filtered Summary

This example demonstrates:

1. A loop between generator and reviewer (which exits when reviewer says “APPROVE”)

2. A summarizer agent that only sees the first user input and the last reviewer message

In [5]:
from autogen_agentchat.agents import AssistantAgent, MessageFilterAgent, MessageFilterConfig, PerSourceFilter
from autogen_agentchat.teams import (
    DiGraphBuilder,
    GraphFlow,
)
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Agents
generator = AssistantAgent("generator", model_client=model_client, system_message="Generate a list of creative ideas.")
reviewer = AssistantAgent(
    "reviewer",
    model_client=model_client,
    system_message="Review ideas and provide feedbacks, or just 'APPROVE' for final approval.",
)
summarizer_core = AssistantAgent(
    "summary", model_client=model_client, system_message="Summarize the user request and the final feedback."
)

# Filtered summarizer
filtered_summarizer = MessageFilterAgent(
    name="summary",
    wrapped_agent=summarizer_core,
    filter=MessageFilterConfig(
        per_source=[
            PerSourceFilter(source="user", position="first", count=1),
            PerSourceFilter(source="reviewer", position="last", count=1),
        ]
    ),
)

# Build graph with conditional loop
builder = DiGraphBuilder()
builder.add_node(generator).add_node(reviewer).add_node(filtered_summarizer)
builder.add_edge(generator, reviewer)
builder.add_edge(reviewer, filtered_summarizer, condition=lambda msg: "APPROVE" in msg.to_model_text())
builder.add_edge(reviewer, generator, condition=lambda msg: "APPROVE" not in msg.to_model_text())
builder.set_entry_point(generator)  # Set entry point to generator. Required if there are no source nodes.
graph = builder.build()

termination_condition = MaxMessageTermination(10)

# Create the flow
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
    termination_condition=termination_condition
)

# Run the flow and pretty print the output in the console
await Console(flow.run_stream(task="Brainstorm ways to reduce plastic waste."))

---------- TextMessage (user) ----------
Brainstorm ways to reduce plastic waste.
---------- TextMessage (generator) ----------
Here's a comprehensive list of ways to reduce plastic waste, broken down by different spheres of influence:

## Individual Actions (The "R"s)

1.  **Refuse Single-Use Plastics:**
    *   **Reusable Bags:** Always carry reusable shopping bags.
    *   **Reusable Bottles/Cups:** Bring your own water bottle and coffee cup.
    *   **Say No to Straws:** Decline plastic straws, or bring a reusable one (metal, bamboo, glass).
    *   **Reusable Cutlery:** Carry a set of reusable utensils for takeout/lunch.
    *   **Avoid Excess Packaging:** Choose products with minimal or no plastic packaging (e.g., unpackaged produce).
    *   **Refuse Promotional Items:** Decline plastic trinkets, pens, or freebies.

2.  **Reduce Plastic Consumption:**
    *   **Buy in Bulk:** Purchase food, cleaning supplies, and personal care items from bulk bins using your own containers.
    

TaskResult(messages=[TextMessage(id='3dae8760-6881-449f-9a17-a7d4c8af22d3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 24, 31, 625424, tzinfo=datetime.timezone.utc), content='Brainstorm ways to reduce plastic waste.', type='TextMessage'), TextMessage(id='0e33869a-6713-4470-8431-3a0eb2edd621', source='generator', models_usage=RequestUsage(prompt_tokens=17, completion_tokens=1789), metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 24, 49, 915638, tzinfo=datetime.timezone.utc), content='Here\'s a comprehensive list of ways to reduce plastic waste, broken down by different spheres of influence:\n\n## Individual Actions (The "R"s)\n\n1.  **Refuse Single-Use Plastics:**\n    *   **Reusable Bags:** Always carry reusable shopping bags.\n    *   **Reusable Bottles/Cups:** Bring your own water bottle and coffee cup.\n    *   **Say No to Straws:** Decline plastic straws, or bring a reusable one (metal, bamboo, glass).\n    *   **Reusable 

Advanced Example: Cycles With Activation Group Examples

The following examples demonstrate how to use activation_group and activation_condition to handle complex dependency patterns in cyclic graphs, especially when multiple paths lead to the same target node.

Example 1: Loop with Multiple Paths - “All” Activation (A→B→C→B)
In this scenario, we have A → B → C → B, where B has two incoming edges (from A and from C). By default, B requires all its dependencies to be satisfied before executing.

This example shows a review loop where both the initial input (A) and the feedback (C) must be processed before B can execute again.

In [7]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Model client
client = model_client

# Create agents for A→B→C→B→E scenario
agent_a = AssistantAgent("A", model_client=client, system_message="Start the process and provide initial input.")
agent_b = AssistantAgent(
    "B",
    model_client=client,
    system_message="Process input from A or feedback from C. Say 'CONTINUE' if it's from A or 'STOP' if it's from C.",
)
agent_c = AssistantAgent("C", model_client=client, system_message="Review B's output and provide feedback.")
agent_e = AssistantAgent("E", model_client=client, system_message="Finalize the process.")

# Build the graph with activation groups
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c).add_node(agent_e)

# A → B (initial path)
builder.add_edge(agent_a, agent_b, activation_group="initial")

# B → C
builder.add_edge(agent_b, agent_c, condition="CONTINUE")

# C → B (loop back - different activation group)
builder.add_edge(agent_c, agent_b, activation_group="feedback")

# B → E (exit condition)
builder.add_edge(agent_b, agent_e, condition="STOP")

termination_condition = MaxMessageTermination(10)
# Build and create flow
graph = builder.build()
flow = GraphFlow(participants=[agent_a, agent_b, agent_c, agent_e], graph=graph, termination_condition=termination_condition)

print("=== Example 1: A→B→C→B with 'All' Activation ===")
print("B will exit when it receives a message from C")
await Console(flow.run_stream(task="Start a review process for a document."))

=== Example 1: A→B→C→B with 'All' Activation ===
B will exit when it receives a message from C
---------- TextMessage (user) ----------
Start a review process for a document.


---------- TextMessage (A) ----------
Okay, let's start a review process for a document. To make this effective, I need a little more information from you.

**Please provide the following details to help me tailor the review process:**

1.  **Document Title/Type:** What kind of document is it? (e.g., Marketing Brochure, Technical Report, Legal Contract, Policy Document, Project Plan, Blog Post, Software Requirements Specification, etc.)
2.  **Purpose of the Document:** What is it intended to achieve? (e.g., inform customers, guide employees, define a project, sell a product, secure approval, etc.)
3.  **Target Audience:** Who is the primary reader of this document? (e.g., internal team, clients, general public, executives, technical experts, etc.)
4.  **Current State of the Document:** Is it a first draft, a near-final version, a minor update to an existing document?
5.  **Primary Goal of This Review:** What specifically do you want to achieve with this review?
    *   Catch errors (ty

TaskResult(messages=[TextMessage(id='89c47dbd-6a85-42c4-b446-3611b5f37c72', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 30, 2, 770741, tzinfo=datetime.timezone.utc), content='Start a review process for a document.', type='TextMessage'), TextMessage(id='37ce42d0-c932-400f-8975-f4506a416fc9', source='A', models_usage=RequestUsage(prompt_tokens=18, completion_tokens=1213), metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 30, 14, 753287, tzinfo=datetime.timezone.utc), content='Okay, let\'s start a review process for a document. To make this effective, I need a little more information from you.\n\n**Please provide the following details to help me tailor the review process:**\n\n1.  **Document Title/Type:** What kind of document is it? (e.g., Marketing Brochure, Technical Report, Legal Contract, Policy Document, Project Plan, Blog Post, Software Requirements Specification, etc.)\n2.  **Purpose of the Document:** What is it intended t

Example 2: Loop with Multiple Paths - “Any” Activation (A→B→(C1,C2)→B)

In this more complex scenario, we have A → B → (C1, C2) → B, where:

1. B fans out to both C1 and C2 in parallel

2. Both C1 and C2 feed back to B

3. B uses “any” activation, meaning it executes as soon as either C1 or C2 completes

This is useful for scenarios where you want the fastest response to trigger the next step.

In [ ]:
# Create agents for A→B→(C1,C2)→B scenario
agent_a2 = AssistantAgent("A", model_client=client, system_message="Initiate a task that needs parallel processing.")
agent_b2 = AssistantAgent(
    "B",
    model_client=client,
    system_message="Coordinate parallel tasks. Say 'PROCESS' to start parallel work or 'DONE' to finish.",
)
agent_c1 = AssistantAgent("C1", model_client=client, system_message="Handle task type 1. Say 'C1_COMPLETE' when done.")
agent_c2 = AssistantAgent("C2", model_client=client, system_message="Handle task type 2. Say 'C2_COMPLETE' when done.")
agent_e = AssistantAgent("E", model_client=client, system_message="Finalize the process.")

# Build the graph with "any" activation
builder2 = DiGraphBuilder()
builder2.add_node(agent_a2).add_node(agent_b2).add_node(agent_c1).add_node(agent_c2).add_node(agent_e)

# A → B (initial)
builder2.add_edge(agent_a2, agent_b2)

# B → C1 and B → C2 (parallel fan-out)
builder2.add_edge(agent_b2, agent_c1, condition="PROCESS")
builder2.add_edge(agent_b2, agent_c2, condition="PROCESS")

# B → E (exit condition)
builder2.add_edge(agent_b2, agent_e, condition=lambda msg: "DONE" in msg.to_model_text())

# C1 → B and C2 → B (both in same activation group with "any" condition)
builder2.add_edge(
    agent_c1, agent_b2, activation_group="loop_back_group", activation_condition="any", condition="C1_COMPLETE"
)

builder2.add_edge(
    agent_c2, agent_b2, activation_group="loop_back_group", activation_condition="any", condition="C2_COMPLETE"
)

# Build and create flow
graph2 = builder2.build()
flow2 = GraphFlow(participants=[agent_a2, agent_b2, agent_c1, agent_c2, agent_e], graph=graph2)

print("=== Example 2: A→B→(C1,C2)→B with 'Any' Activation ===")
print("B will execute as soon as EITHER C1 OR C2 completes (whichever finishes first)")
await Console(flow2.run_stream(task="Start a parallel processing task."))

=== Example 2: A→B→(C1,C2)→B with 'Any' Activation ===
B will execute as soon as EITHER C1 OR C2 completes (whichever finishes first)


Example 3: Mixed Activation Groups

This example shows how different activation groups can coexist in the same graph. We have a scenario where:

1. Node D receives inputs from multiple sources with different activation requirements

2. Some dependencies use “all” activation (must wait for all inputs)

3. Other dependencies use “any” activation (proceed on first input)

This pattern is useful for complex workflows where different types of dependencies have different urgency levels.

In [12]:
# Create agents for mixed activation scenario
agent_a3 = AssistantAgent("A", model_client=client, system_message="Provide critical input that must be processed.")
agent_b3 = AssistantAgent("B", model_client=client, system_message="Provide secondary critical input.")
agent_c3 = AssistantAgent("C", model_client=client, system_message="Provide optional quick input.")
agent_d3 = AssistantAgent("D", model_client=client, system_message="Process inputs based on different priority levels.")

# Build graph with mixed activation groups
builder3 = DiGraphBuilder()
builder3.add_node(agent_a3).add_node(agent_b3).add_node(agent_c3).add_node(agent_d3)

# Critical inputs that must ALL be present (activation_group="critical", activation_condition="all")
builder3.add_edge(agent_a3, agent_d3, activation_group="critical", activation_condition="all")
builder3.add_edge(agent_b3, agent_d3, activation_group="critical", activation_condition="all")

# Optional input that can trigger execution on its own (activation_group="optional", activation_condition="any")
builder3.add_edge(agent_c3, agent_d3, activation_group="optional", activation_condition="any")

# Build and create flow
graph3 = builder3.build()
flow3 = GraphFlow(participants=[agent_a3, agent_b3, agent_c3, agent_d3], graph=graph3)

print("=== Example 3: Mixed Activation Groups ===")
print("D will execute when:")
print("- BOTH A AND B complete (critical group with 'all' activation), OR")
print("- C completes (optional group with 'any' activation)")
print("This allows for both required dependencies and fast-path triggers.")
await Console(flow3.run_stream(task="Process inputs with mixed priority levels."))

=== Example 3: Mixed Activation Groups ===
D will execute when:
- BOTH A AND B complete (critical group with 'all' activation), OR
- C completes (optional group with 'any' activation)
This allows for both required dependencies and fast-path triggers.
---------- TextMessage (user) ----------
Process inputs with mixed priority levels.
---------- TextMessage (B) ----------
Processing inputs with mixed priority levels is a fundamental challenge in virtually every domain, from personal task management to complex enterprise systems and emergency response. The goal is always to optimize resource allocation, ensure critical items are addressed, and maintain overall efficiency and effectiveness.

---

### **Secondary Critical Input:**

The single most critical input for effectively processing mixed-priority items is the **establishment of clear, agreed-upon, and measurable prioritization criteria.** Without this, any attempt at sorting or sequencing inputs will be arbitrary, inconsistent, and p

TaskResult(messages=[TextMessage(id='29ad2e7b-650b-4272-97ab-9ae52b7b032f', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 40, 11, 290418, tzinfo=datetime.timezone.utc), content='Process inputs with mixed priority levels.', type='TextMessage'), TextMessage(id='1ef681f1-ec46-4248-85a6-e10ea6c87ce2', source='B', models_usage=RequestUsage(prompt_tokens=14, completion_tokens=1514), metadata={}, created_at=datetime.datetime(2025, 12, 8, 17, 40, 25, 860722, tzinfo=datetime.timezone.utc), content='Processing inputs with mixed priority levels is a fundamental challenge in virtually every domain, from personal task management to complex enterprise systems and emergency response. The goal is always to optimize resource allocation, ensure critical items are addressed, and maintain overall efficiency and effectiveness.\n\n---\n\n### **Secondary Critical Input:**\n\nThe single most critical input for effectively processing mixed-priority items is the **